<a href="https://colab.research.google.com/github/krishnakanthkona-3110/Interview_Prep/blob/main/Interview_Focused/Deployment_Related.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#AWS Deployment for RAG Application Interview Preparation Guide


This guide prepares you to explain an AWS-deployed RAG application in interviews. The application exposes FastAPI endpoints through an API Gateway, executes the application in AWS Lambda using a Docker container image stored in Amazon ECR, retrieves relevant vectors from Qdrant Cloud, and uses an LLM/embedding model through the configured model provider. CloudWatch is used for AWS logs and operational monitoring, while Langfuse is used for LLM/RAG observability such as traces, latency and token/usage information. A CI/CD pipeline automates build, test, image publishing and deployment.

# 1) What is CI/CD Pipeline?
 - CI means Continuous Integration and CD means Continuous Delivery or Deployment

 -  It is a series of automated steps that take our code from commit to production.

 - First developer pushes the code to GitHub. GitHub actions comes into play.

 # 2) What is GitHub actions?
 - It's a GitHub service that automatically runs the steps we define.
 - So now, GitHub Actions start our CI/CD process.

 # 3) What comes under CI?
 - Build and Test

 #  Build:
 - Build means turning our source code into something that can be run.

# CI:
Code → Test → Build Docker Image → Push to ECR

# CD:
ECR Image → Deploy to Staging → Approval → Production

In [ ]:
'''

1. Developer writes code
        ↓
2. Push / Merge code to GitHub main
        ↓
3. GitHub Actions pipeline starts
        ↓
4. Install dependencies for TESTING
        ↓
5. Run automated tests
        ↓
   ❌ FAIL → STOP
        ↓
   ✅ PASS
        ↓
6. Build Docker image
        ↓
7. Push Docker image to ECR
        ↓
8. Update Lambda with that image
        ↓
9. Publish new Lambda version
        ↓
10. Point STAGING alias to new version
        ↓
11. Run smoke tests against staging
        ↓
    ❌ FAIL → STOP / rollback
        ↓
    ✅ PASS
        ↓
12. Point PRODUCTION alias to new version
        ↓
13. Production deployment complete

'''

'\n\n1. Developer writes code\n        ↓\n2. Push / Merge code to GitHub main\n        ↓\n3. GitHub Actions pipeline starts\n        ↓\n4. Install dependencies for TESTING\n        ↓\n5. Run automated tests\n        ↓\n   ❌ FAIL → STOP\n        ↓\n   ✅ PASS\n        ↓\n6. Build Docker image\n        ↓\n7. Push Docker image to ECR\n        ↓\n8. Update Lambda with that image\n        ↓\n9. Publish new Lambda version\n        ↓\n10. Point STAGING alias to new version\n        ↓\n11. Run smoke tests against staging\n        ↓\n    ❌ FAIL → STOP / rollback\n        ↓\n    ✅ PASS\n        ↓\n12. Point PRODUCTION alias to new version\n        ↓\n13. Production deployment complete\n\n'

# Step-4 and 5:
After the code is pushed or merged, GitHub Actions installs the dependencies and runs our automated tests. We validate core application functionality, API behavior, and integration points. If the tests fail, the pipeline stops and the Docker image is not promoted for deployment. Only after the tests pass we proceed with the Docker build and deployment stages.

# Step-6: Build Docker image
Docker reads the Dockerfile, starts from the specified base image, installs the required dependencies, copies the application code and configuration into the image, and creates a Docker image that contains everything required to run the application

# The easiest way to remember

Think of your pipeline as
# TEST → BUILD → STORE → DEPLOY → VERIFY → RELEASE.

In [ ]:
'''

TEST
 ↓
"Is my code working?"

BUILD
 ↓
"Can I package it into a deployable Docker image?"

STORE
 ↓
"Put the image in ECR."

DEPLOY
 ↓
"Put that image into Lambda."

VERIFY
 ↓
"Does the deployed application actually work?"

RELEASE
 ↓
"Send production traffic to it."

'''

'\n\nTEST\n ↓\n"Is my code working?"\n\nBUILD\n ↓\n"Can I package it into a deployable Docker image?"\n\nSTORE\n ↓\n"Put the image in ECR."\n\nDEPLOY\n ↓\n"Put that image into Lambda."\n\nVERIFY\n ↓\n"Does the deployed application actually work?"\n\nRELEASE\n ↓\n"Send production traffic to it."\n\n'

# Step by step for explanation -

I implemented CI/CD using GitHub Actions. Whenever a change is merged into the main branch, the pipeline is triggered automatically.

First, it runs automated tests and code-quality checks. If they pass, it builds a Docker image and pushes it to Amazon ECR, tagging the image with the Git commit SHA so every deployment is traceable to an exact source version.

The pipeline then updates AWS Lambda with that specific container image and publishes a new immutable Lambda version. We deploy that version behind a staging alias and run smoke tests. Only if those tests pass do we move the production alias to the new Lambda version.

This also makes rollback simple: if the new version has an issue, we can point the production alias back to the previous Lambda version without rebuilding the image.

For AWS authentication, GitHub Actions uses OIDC to assume an IAM role, so we don't store long-lived AWS access keys or secrets in GitHub.”

Docker → builds the image

ECR → stores the image

Lambda → runs the image

# Docker -

- Docker packages the application code and its dependencies so it can run consistently anywhere

- Docker packages an application, its dependencies, and runtime environment into a container so it can run consistently across different environments

- Docker containers share the host OS kernel, unlike virtual machines, which include a complete guest operating system.



- docker build → creates an image
- docker run → creates and starts a container
- docker ps → shows running containers

- Docker Compose is a tool used to define and run multiple Docker containers as a single application using a YAML configuration file

- docker compose up → start services
- docker compose down → stop services
- docker compose ps → check services
- docker compose logs → view logs

# AWS ECR-
Amazon ECR is a registry for storing docker images in AWS

# Lambda:
What:

AWS Lambda is a serverless compute service that runs our application code without us managing servers. It can also run applications packaged as Docker images.

Why:

We use Lambda because it removes server-management overhead and automatically scales based on incoming requests. It is suitable for request-driven APIs when the application fits within Lambda's limits.


- Lambda is a serverless service that runs our application without us managing servers and automatically scales based on requests.



# Cold Start:

Cold start is the additional latency that occurs when Lambda creates and initializes a new execution environment before handling a request. Once the environment is warm, Lambda can reuse it for subsequent requests, reducing the initialization overhead.

Easy to remember:

Cold = new environment → initialization → extra latency

Warm = existing environment → reuse → faster response

# CloudWatch — "Is my system healthy?"

CloudWatch monitors the infrastructure — is Lambda running without errors, how long are invocations taking, is API Gateway returning 5xx errors, are we getting throttled. It's the AWS-native layer that tells me if something's broken at the system level — servers, requests, latency, uptime.


# Langfuse — "Is my AI giving good answers?

"Langfuse is specifically for the LLM/RAG side. It traces each individual query — what was retrieved from Qdrant, what prompt got sent to Bedrock, what the model answered, how many tokens it used, and what that cost. So when an answer is wrong or a user complains, I can pull up that exact trace and see whether it was a retrieval problem — pulled the wrong page — or a generation problem — the model didn't use the context correctly."

#Observability — "Can I see what's happening?"

Observability is the ability to understand the health, performance, and behavior of a system using the data it produces.

It mainly uses:

Logs → What happened?

Metrics → How much/how often?

Traces → Where did the request spend time?

Alerts → Is something going wrong?

"Observability means I can look into the system from the outside and understand its state — metrics, logs, traces — without having to guess. In my project, that's CloudWatch for infra (latency, errors, throttling) and Langfuse for the LLM layer (what was retrieved, what was generated, token cost)."

Simple line: "Can I see what's going on inside the system right now?"


In [3]:
'''                 OBSERVABILITY
                      │
        ┌─────────────┼─────────────┐
        ↓             ↓             ↓
      Logs         Metrics        Traces
        │             │             │
     Errors        Latency       Request flow
     Events        Throughput     Service calls
     Messages      Error rate     Dependencies


'''

'                 OBSERVABILITY\n                      │\n        ┌─────────────┼─────────────┐\n        ↓             ↓             ↓\n      Logs         Metrics        Traces\n        │             │             │\n     Errors        Latency       Request flow\n     Events        Throughput     Service calls\n     Messages      Error rate     Dependencies\n\n\n'

#Traceability — "Can I follow one request end to end?"
- Traceability means being able to follow a specific request or transaction through multiple components.

"Traceability is narrower — it's the ability to follow one specific request through every step it touched, in order. In my system, Langfuse gives me that: for a single user query, I can see the exact retrieval call to Qdrant, the exact prompt sent to Bedrock, the exact response, and the token cost — all linked to that one trace ID."